In [18]:
import pandas as pd
from nba_api.live.nba.endpoints.playbyplay import PlayByPlay

gameId = "0022200001"

pbp = PlayByPlay(game_id=gameId)
df = pd.DataFrame(pbp.actions.get_dict())

df.head(5)

,actionNumber,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,y,possession,scoreHome,scoreAway,edited,orderNumber,xLegacy,yLegacy,isFieldGoal,side,description,personIdsFilter,teamId,teamTricode,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerName,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,area,areaDetail,shotDistance,shotResult,blockPlayerName,blockPersonId,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,officialId,turnoverTotal,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,stealPlayerName,stealPersonId
0,2,PT12M00.00S,2022-10-18T23:35:25.4Z,1,REGULAR,period,start,[],0,NaN,NaN,0,0,0,2022-10-18T23:35:25Z,20000,NaN,NaN,0,None,Period Start,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,PT11M57.00S,2022-10-18T23:35:27.8Z,1,REGULAR,jumpball,recovered,[],202699,NaN,NaN,1610612755,0,0,2022-10-18T23:35:27Z,40000,NaN,NaN,0,None,Jump Ball J. Embiid vs. A. Horford: Tip to T. ...,"[202699, 203954, 201143]",1.610613e+09,PHI,startperiod,T. Harris,202699.0,Harris,T. Harris,Embiid,203954.0,Horford,201143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,PT11M38.00S,2022-10-18T23:35:46.0Z,1,REGULAR,2pt,Jump Shot,[],203954,10.857424,73.529412,1610612755,0,0,2022-10-18T23:53:28Z,70000,-118.0,50.0,1,left,MISS J. Embiid 12' turnaround fadeaway Shot - ...,"[203954, 1627759]",1.610613e+09,PHI,turnaround fadeaway,NaN,NaN,Embiid,J. Embiid,NaN,NaN,NaN,NaN,Mid-Range,8-16 Left,12.76,Missed,Brown,1627759.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,PT11M38.00S,2022-10-18T23:35:46.0Z,1,REGULAR,block,,[],1627759,NaN,NaN,1610612755,0,0,2022-10-18T23:35:49Z,80000,NaN,NaN,0,None,J. Brown BLOCK (1 BLK),[1627759],1.610613e+09,BOS,NaN,NaN,NaN,Brown,J. Brown,NaN,NaN,NaN,NaN,Mid-Range,8-16 Left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9,PT11M35.00S,2022-10-18T23:35:49.0Z,1,REGULAR,rebound,offensive,[],200782,NaN,NaN,1610612755,0,0,2022-10-18T23:36:01Z,90000,NaN,NaN,0,None,P. Tucker REBOUND (Off:1 Def:0),[200782],1.610613e+09,PHI,NaN,NaN,NaN,Tucker,P. Tucker,NaN,NaN,NaN,NaN,Mid-Range,8-16 Left,NaN,NaN,NaN,NaN,7.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
from pathlib import Path
import requests


def clockToSeconds(clockValue):
    if pd.isna(clockValue):
        return pd.NA

    clockText = str(clockValue).replace('PT', '').replace('S', '')
    minutesText, secondsText = clockText.split('M', 1)
    return int(minutesText) * 60 + float(secondsText)


def secondsLeftInGame(periodValue, clockValue, periodTypeValue='REGULAR'):
    clockSeconds = clockToSeconds(clockValue)

    if str(periodTypeValue).upper() == 'OVERTIME' or periodValue > 4:
        overtimeNumber = max(periodValue - 5, 0)
        return overtimeNumber * 300 + clockSeconds

    return max(4 - periodValue, 0) * 720 + clockSeconds


def loadGameMeta(gameIdValue):
    candidateDirs = [Path('data/nba_gamelog'), Path('nba_gamelog')]
    gameIdValue = str(gameIdValue)

    gamelogPaths = []
    for candidateDir in candidateDirs:
        gamelogPaths.extend(sorted(candidateDir.glob('gamelog_*.parquet'), reverse=True))

    for gamelogPath in gamelogPaths:
        gamelogDf = pd.read_parquet(gamelogPath)
        gamelogDf.columns = gamelogDf.columns.str.lower().str.replace(r'_(\w)', lambda m: m.group(1).upper(), regex=True)
        gamelogDf['gameId'] = gamelogDf['gameId'].astype(str)

        gameRows = gamelogDf.loc[gamelogDf['gameId'] == gameIdValue, ['season', 'gameId', 'gameDate', 'matchup', 'teamId', 'teamAbbreviation', 'wl']].copy()
        if gameRows.empty:
            continue

        homeRow = gameRows.loc[gameRows['matchup'].str.contains(' vs. ', na=False)].head(1)
        awayRow = gameRows.loc[gameRows['matchup'].str.contains(' @ ', na=False)].head(1)

        if homeRow.empty:
            continue

        homeRow = homeRow.iloc[0]
        awayAbbreviation = awayRow.iloc[0]['teamAbbreviation'] if not awayRow.empty else homeRow['matchup'].split(' vs. ')[1]
        awayTeamId = awayRow.iloc[0]['teamId'] if not awayRow.empty else pd.NA

        return {
            'season': homeRow['season'],
            'gameId': homeRow['gameId'],
            'gameDate': pd.to_datetime(homeRow['gameDate']).date(),
            'matchup': homeRow['matchup'],
            'homeTeamId': homeRow['teamId'],
            'homeAbbreviation': homeRow['teamAbbreviation'],
            'awayTeamId': awayTeamId,
            'awayAbbreviation': awayAbbreviation,
            'homeWin': {'W': 1, 'L': 0}.get(homeRow['wl'], pd.NA)
        }

    searchedDirs = ', '.join(str(path) for path in candidateDirs)
    raise FileNotFoundError(f'No local gamelog entry found for gameId {gameIdValue} in {searchedDirs}')


df = df.copy()
df['gameId'] = str(gameId)
df['actionId'] = df['actionNumber']

for scoreColumn in ['scoreHome', 'scoreAway']:
    df[scoreColumn] = pd.to_numeric(df[scoreColumn], errors='coerce').ffill()

df['pointsTotal'] = df['scoreHome'] + df['scoreAway']
df['quarter'] = df['period']
df['secondsLeft'] = df.apply(
    lambda row: secondsLeftInGame(row['period'], row['clock'], row.get('periodType', 'REGULAR')),
    axis=1
)

gameMeta = loadGameMeta(gameId)
for columnName, columnValue in gameMeta.items():
    df[columnName] = columnValue

df['scoreDif'] = df['scoreHome'] - df['scoreAway']

if 'teamId' in df.columns:
    df['teamId'] = pd.to_numeric(df['teamId'], errors='coerce').astype('Int64')
    df['homeTeamId'] = pd.to_numeric(df['homeTeamId'], errors='coerce').astype('Int64')
    df['awayTeamId'] = pd.to_numeric(df['awayTeamId'], errors='coerce').astype('Int64')
    df['isHomeAction'] = df['teamId'].eq(df['homeTeamId'])
    df.loc[df['teamId'].isna(), 'isHomeAction'] = pd.NA
    df['actionTeamSide'] = pd.Series(pd.NA, index=df.index, dtype='object')
    df.loc[df['teamId'].eq(df['homeTeamId']), 'actionTeamSide'] = 'home'
    df.loc[df['teamId'].eq(df['awayTeamId']), 'actionTeamSide'] = 'away'

if 'possession' in df.columns:
    df['possession'] = pd.to_numeric(df['possession'], errors='coerce').astype('Int64')
    df['isHomePossession'] = df['possession'].eq(df['homeTeamId'])
    df.loc[df['possession'].isna(), 'isHomePossession'] = pd.NA
    df['possessionTeamSide'] = pd.Series(pd.NA, index=df.index, dtype='object')
    df.loc[df['possession'].eq(df['homeTeamId']), 'possessionTeamSide'] = 'home'
    df.loc[df['possession'].eq(df['awayTeamId']), 'possessionTeamSide'] = 'away'

preferredColumns = [
    'season',
    'gameId',
    'gameDate',
    'matchup',
    'homeTeamId',
    'homeAbbreviation',
    'awayTeamId',
    'awayAbbreviation',
    'homeWin',
    'actionId',
    'description',
    'quarter',
    'secondsLeft',
    'scoreHome',
    'scoreAway',
    'scoreDif',
    'pointsTotal',
    'teamId',
    'teamTricode',
    'actionTeamSide',
    'isHomeAction',
    'possession',
    'possessionTeamSide',
    'isHomePossession'
]

existingPreferredColumns = [column for column in preferredColumns if column in df.columns]
remainingColumns = [column for column in df.columns if column not in existingPreferredColumns]
df = df[existingPreferredColumns + remainingColumns].copy()

rotowireUrl = 'https://www.rotowire.com/betting/nba/tables/games-archive.php'
try:
    rotowireResponse = requests.get(rotowireUrl, timeout=30)
    rotowireResponse.raise_for_status()
    rotowireDf = pd.DataFrame(rotowireResponse.json())
    rotowireDf = rotowireDf[['game_date', 'home_team_abbrev', 'visit_team_abbrev', 'line']].rename(columns={
        'game_date': 'gameDate',
        'home_team_abbrev': 'homeAbbreviation',
        'visit_team_abbrev': 'awayAbbreviation'
    })
    rotowireDf['gameDate'] = pd.to_datetime(rotowireDf['gameDate']).dt.date
    rotowireDf = rotowireDf.drop_duplicates(subset=['gameDate', 'homeAbbreviation', 'awayAbbreviation'])
    df = df.merge(rotowireDf, on=['gameDate', 'homeAbbreviation', 'awayAbbreviation'], how='left')
except Exception as exc:
    print(f'Rotowire merge skipped: {exc}')

df.head()


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,actionId,description,quarter,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,teamId,teamTricode,actionTeamSide,isHomeAction,possession,possessionTeamSide,isHomePossession,actionNumber,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,y,edited,orderNumber,xLegacy,yLegacy,isFieldGoal,side,personIdsFilter,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerName,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,area,areaDetail,shotDistance,shotResult,blockPlayerName,blockPersonId,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,officialId,turnoverTotal,assistPlayerNameInitial,assistPersonId,assistTotal,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,stealPlayerName,stealPersonId,line
0,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,2,Period Start,1,2880.0,0,0,0,0,<NA>,NaN,<NA>,<NA>,0,<NA>,False,2,PT12M00.00S,2022-10-18T23:35:25.4Z,1,REGULAR,period,start,[],0,NaN,NaN,2022-10-18T23:35:25Z,20000,NaN,NaN,0,None,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
1,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,4,Jump Ball J. Embiid vs. A. Horford: Tip to T. ...,1,2877.0,0,0,0,0,1610612755,PHI,away,False,1610612755,away,False,4,PT11M57.00S,2022-10-18T23:35:27.8Z,1,REGULAR,jumpball,recovered,[],202699,NaN,NaN,2022-10-18T23:35:27Z,40000,NaN,NaN,0,None,"[202699, 203954, 201143]",startperiod,T. Harris,202699.0,Harris,T. Harris,Embiid,203954.0,Horford,201143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
2,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,7,MISS J. Embiid 12' turnaround fadeaway Shot - ...,1,2858.0,0,0,0,0,1610612755,PHI,away,False,1610612755,away,False,7,PT11M38.00S,2022-10-18T23:35:46.0Z,1,REGULAR,2pt,Jump Shot,[],203954,10.857424,73.529412,2022-10-18T23:53:28Z,70000,-118.0,50.0,1,left,"[203954, 1627759]",turnaround fadeaway,NaN,NaN,Embiid,J. Embiid,NaN,NaN,NaN,NaN,Mid-Range,8-16 Left,12.76,Missed,Brown,1627759.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
3,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,8,J. Brown BLOCK (1 BLK),1,2858.0,0,0,0,0,1610612738,BOS,home,True,1610612755,away,False,8,PT11M38.00S,2022-10-18T23:35:46.0Z,1,REGULAR,block,,[],1627759,NaN,NaN,2022-10-18T23:35:49Z,80000,NaN,NaN,0,None,[1627759],NaN,NaN,NaN,Brown,J. Brown,NaN,NaN,NaN,NaN,Mid-Range,8-16 Left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
4,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,9,P. Tucker REBOUND (Off:1 Def:0),1,2855.0,0,0,0,0,1610612755,PHI,away,False,1610612755,away,False,9,PT11M35.00S,2022-10-18T23:35:49.0Z,1,REGULAR,rebound,offensive,[],200782,NaN,NaN,2022-10-18T23:36:01Z,90000,NaN,NaN,0,None,[200782],NaN,NaN,NaN,Tucker,P. Tucker,NaN,NaN,NaN,NaN,Mid-Range,8-16 Left,NaN,NaN,NaN,NaN,7.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0


In [20]:
df.columns

Index(['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId',
       'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin',
       'actionId', 'description', 'quarter', 'secondsLeft', 'scoreHome',
       'scoreAway', 'scoreDif', 'pointsTotal', 'teamId', 'teamTricode',
       'actionTeamSide', 'isHomeAction', 'possession', 'possessionTeamSide',
       'isHomePossession', 'actionNumber', 'clock', 'timeActual', 'period',
       'periodType', 'actionType', 'subType', 'qualifiers', 'personId', 'x',
       'y', 'edited', 'orderNumber', 'xLegacy', 'yLegacy', 'isFieldGoal',
       'side', 'personIdsFilter', 'descriptor', 'jumpBallRecoveredName',
       'jumpBallRecoverdPersonId', 'playerName', 'playerNameI',
       'jumpBallWonPlayerName', 'jumpBallWonPersonId',
       'jumpBallLostPlayerName', 'jumpBallLostPersonId', 'area', 'areaDetail',
       'shotDistance', 'shotResult', 'blockPlayerName', 'blockPersonId',
       'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal',
   

In [21]:
cols_to_drop = ['isHomePossession', 'teamTricode', 'teamId', 'isHomeAction', 'clock', 'timeActual', 
                'periodType', 'qualifiers', 'x', 'y', 'edited', 'orderNumber', 'xLegacy', 'yLegacy', 
                'isFieldGoal', 'side', 'personIdsFilter', 'descriptor', 'jumpBallRecoveredName',
       'jumpBallRecoverdPersonId', 'playerNameI',
       'jumpBallWonPlayerName', 'jumpBallWonPersonId',
       'jumpBallLostPlayerName', 'jumpBallLostPersonId', 'area', 'areaDetail',
       'shotDistance','blockPlayerName', 'blockPersonId',
       'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal',
       'reboundOffensiveTotal', 'officialId', 'turnoverTotal',
       'assistPlayerNameInitial', 'assistPersonId', 'assistTotal','foulDrawnPlayerName',
       'foulDrawnPersonId', 'stealPlayerName', 'stealPersonId', 'actionId', 'actionNumber',
       'period']
df = df.drop(columns=cols_to_drop)

In [22]:
df.head()

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,possessionTeamSide,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,line
0,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,Period Start,1,2880.0,0,0,0,0,<NA>,0,<NA>,period,start,0,NaN,NaN,NaN,NaN,-3.0
1,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,Jump Ball J. Embiid vs. A. Horford: Tip to T. ...,1,2877.0,0,0,0,0,away,1610612755,away,jumpball,recovered,202699,Harris,NaN,NaN,NaN,-3.0
2,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,MISS J. Embiid 12' turnaround fadeaway Shot - ...,1,2858.0,0,0,0,0,away,1610612755,away,2pt,Jump Shot,203954,Embiid,Missed,NaN,NaN,-3.0
3,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,J. Brown BLOCK (1 BLK),1,2858.0,0,0,0,0,home,1610612755,away,block,,1627759,Brown,NaN,NaN,NaN,-3.0
4,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,P. Tucker REBOUND (Off:1 Def:0),1,2855.0,0,0,0,0,away,1610612755,away,rebound,offensive,200782,Tucker,NaN,NaN,NaN,-3.0


### Need to add:
- possession
- free throws
- ejections
- bonus